# Phase 5 — parent, incumbent, and a threshold that did nothing

A feature step is built from one **parent**, but the stop rule asks whether it beats the best table seen anywhere so far — the **incumbent**. Those are different comparisons on branches. This notebook reads the reconstructed prediction contrasts by hand before looking at the registered verdict.

In [ ]:
import json
from pathlib import Path

import pandas as pd

HERE = Path.cwd()
EXPERIMENT = HERE if (HERE / 'contrasts.csv').exists() else HERE / 'research/archive/experiments/0080-0081-incumbent-threshold'
contrasts = pd.read_csv(EXPERIMENT / 'contrasts.csv')
summary = json.loads((EXPERIMENT / 'summary.json').read_text())
contrasts.shape, summary['high_capacity_labeled_steps']

## 1. How often was the parent not the incumbent?

If this count were zero, recomputing the incumbent contrast would have been bookkeeping. It is not zero: the search branched often enough that the stored parent interval answers another question on a substantial share of steps.

In [ ]:
measured = contrasts.dropna(subset=['incumbent'])
by_run = measured.assign(parent_differs=measured.parent != measured.incumbent).groupby('run').agg(
    contrasts=('candidate', 'size'),
    parent_differs=('parent_differs', 'sum'),
    positive_incumbent_gains=('incumbent_gain', lambda s: (s > 0).sum()),
)
assert int(by_run.parent_differs.sum()) == 20
by_run

## 2. The registered comparison

Decision 0080 fixed `k = 0.25` before these contrasts were computed. It had to improve precision over `gain > 0` and retain at least half of the true high-capacity incumbent gains. Both conditions were required.

In [ ]:
rules = pd.DataFrame(summary['rules']).T[['accepted', 'true_positive', 'truth_positive', 'precision', 'recall']]
assert rules.loc['0.0', 'accepted'] == 11
assert rules.loc['0.25', 'accepted'] == 11
assert rules.loc['0.0', 'precision'] == rules.loc['0.25', 'precision']
assert summary['registered_prediction_passed'] is False
rules

## 3. Why `k = 0.5` is not the answer

The sensitivity rows look better, but they were explicitly barred from replacing a failed primary rule. Looking at individual ratios shows why selecting one now would be fragile: a false positive survives above one block spread, while the smallest real gain sits almost exactly at a quarter.

In [ ]:
labeled = contrasts.dropna(subset=['high_capacity_positive']).copy()
labeled['ratio'] = labeled.incumbent_gain / labeled.incumbent_block_sd
labeled['capacity_positive'] = labeled.high_capacity_positive.astype(str).str.lower().eq('true')
false_positive = labeled[(labeled.incumbent_gain > 0) & ~labeled.capacity_positive]
small_capacity_positive = labeled[labeled.capacity_positive].sort_values('ratio').head(1)
assert false_positive.shape[0] == 4
pd.concat([
    false_positive[['run', 'candidate', 'incumbent_gain', 'incumbent_block_sd', 'ratio', 'capacity_incumbent_gain']],
    small_capacity_positive[['run', 'candidate', 'incumbent_gain', 'incumbent_block_sd', 'ratio', 'capacity_incumbent_gain']],
]).sort_values('ratio')

## 4. The useful object survived; the gate did not

The exact candidate-vs-incumbent card is worth persisting: parent and incumbent differed on 20 of 51 measured steps, and IEEE had blockwise sign disagreement on 20 of 23 incumbent contrasts. But `0.25 × block_sd` changed no capacity-labelled decision. The safe next move is reporting the exact contrast, fixing tied time boundaries, and keeping any future threshold in shadow mode rather than wiring this constant into stopping.